# 02 · Features y división A/B — SepsisWatch Z

**Para qué sirve:** el modelo no entiende "el paciente empeora"; solo ve números por fila.
Este notebook convierte cada **hora de cada paciente** en una fila con variables (*features*) que resumen su historia **hasta esa hora**:
último valor medido, si se midió, hace cuánto, tendencias de 6 horas y criterios clínicos.

**Regla de oro:** cada fila usa solo información de **esa hora o anteriores**. Nunca datos del futuro (en la vida real, a las 10 AM no sabes qué pasará a las 11).

**Entrada:** `~/shared/sepsis/data/processed/sepsis_all.pkl` (de `01_ingesta_exploracion`)
**Salida:** `~/shared/sepsis/data/processed/features.pkl` + `feature_list.json`
→ la usan **Hemsy** (XGBoost) y **Jhojan** (red en NNPA).

## 1. Cargar los datos consolidados

In [1]:
import os, json, time, platform
import numpy as np
import pandas as pd

BASE = os.path.expanduser("~/shared/sepsis/data/processed")
t_start = time.time()
df = pd.read_pickle(f"{BASE}/sepsis_all.pkl")
df = df.sort_values(["patient_id", "hour"], kind="stable").reset_index(drop=True)
print(f"{len(df):,} filas | {df['patient_id'].nunique():,} pacientes | {platform.machine()}")

VITALS = ["HR", "O2Sat", "Temp", "SBP", "MAP", "DBP", "Resp", "EtCO2"]
LABS = ["BaseExcess", "HCO3", "FiO2", "pH", "PaCO2", "SaO2", "AST", "BUN",
        "Alkalinephos", "Calcium", "Chloride", "Creatinine", "Bilirubin_direct",
        "Glucose", "Lactate", "Magnesium", "Phosphate", "Potassium",
        "Bilirubin_total", "TroponinI", "Hct", "Hgb", "PTT", "WBC",
        "Fibrinogen", "Platelets"]
MEASURED = VITALS + LABS
g = df.groupby("patient_id", observed=True, sort=False)

1,552,210 filas | 40,336 pacientes | s390x


## 2. ¿Se midió esta hora? y ¿hace cuántas horas fue la última medición?

- `<var>_medido` = 1 si esa variable se midió **esta hora**. Que el médico pida un examen ya es información.
- `<var>_horas_desde` = horas desde la última medición (vacío si nunca se midió).

In [2]:
t0 = time.time()
feats = {}
measured_hour = pd.DataFrame(index=df.index)
for c in MEASURED:
    m = df[c].notna()
    feats[f"{c}_medido"] = m.astype(np.int8)
    measured_hour[c] = df["hour"].where(m).astype(np.float32)
last_hour = measured_hour.groupby(df["patient_id"], observed=True, sort=False).ffill()
for c in MEASURED:
    feats[f"{c}_horas_desde"] = (df["hour"] - last_hour[c]).astype(np.float32)
del measured_hour, last_hour
print(f"{2 * len(MEASURED)} columnas en {time.time() - t0:.1f} s")

68 columnas en 0.6 s


## 3. Arrastrar el último valor medido (*forward fill*)

Si el lactato se midió a las 3 AM, ese valor se mantiene hasta la siguiente medición.
Se hace **dentro de cada paciente** (nunca se arrastra un valor de un paciente a otro).

In [3]:
t0 = time.time()
filled = g[MEASURED].ffill().astype(np.float32)
for c in MEASURED:
    feats[c] = filled[c]
print(f"{len(MEASURED)} columnas en {time.time() - t0:.1f} s")

34 columnas en 0.3 s


## 4. Tendencias de 6 horas en los signos vitales

Para frecuencia cardíaca, saturación, temperatura, presión sistólica, presión media y frecuencia respiratoria:
**media, mínimo y máximo** de las últimas 6 horas, y **cambio** respecto a hace 6 horas (¿está subiendo o bajando?).
La exploración mostró que **la tendencia importa más que el valor suelto**.

In [4]:
t0 = time.time()
TREND = ["HR", "O2Sat", "Temp", "SBP", "MAP", "Resp"]
roll = filled[TREND].groupby(df["patient_id"], observed=True, sort=False).rolling(6, min_periods=1)
for stat in ["mean", "min", "max"]:
    r = getattr(roll, stat)().reset_index(level=0, drop=True).sort_index()
    for c in TREND:
        feats[f"{c}_6h_{stat}"] = r[c].astype(np.float32)
shifted = filled[TREND].groupby(df["patient_id"], observed=True, sort=False).shift(6)
for c in TREND:
    feats[f"{c}_6h_cambio"] = (filled[c] - shifted[c]).astype(np.float32)
del roll, shifted
print(f"{4 * len(TREND)} columnas en {time.time() - t0:.1f} s")

24 columnas en 10.3 s


## 5. Variables clínicas (lo que usa un médico)

- **Índice de shock** = frecuencia cardíaca / presión sistólica (> 1 es mala señal).
- **SIRS** (0–4): temperatura > 38 o < 36, pulso > 90, respiración > 20 o PaCO2 < 32, leucocitos > 12 o < 4.
- **qSOFA** (0–2 con los datos disponibles): respiración ≥ 22, presión sistólica ≤ 100. *(El tercer criterio, estado mental, no está en el dataset.)*
- **Disfunción de órganos:** lactato > 2, plaquetas < 150, creatinina > 1.2, bilirrubina > 1.2.

SIRS y qSOFA son **la práctica actual**: más adelante los usamos como **línea base** para demostrar que nuestro modelo es mejor.

In [5]:
f = filled
feats["indice_shock"] = (f["HR"] / f["SBP"]).astype(np.float32)
sirs = ((f["Temp"] > 38) | (f["Temp"] < 36)).astype(np.int8) \
     + (f["HR"] > 90).astype(np.int8) \
     + ((f["Resp"] > 20) | (f["PaCO2"] < 32)).astype(np.int8) \
     + ((f["WBC"] > 12) | (f["WBC"] < 4)).astype(np.int8)
feats["SIRS"] = sirs
feats["qSOFA"] = ((f["Resp"] >= 22).astype(np.int8) + (f["SBP"] <= 100).astype(np.int8))
feats["lactato_alto"] = (f["Lactate"] > 2).astype(np.int8)
feats["plaquetas_bajas"] = (f["Platelets"] < 150).astype(np.int8)
feats["creatinina_alta"] = (f["Creatinine"] > 1.2).astype(np.int8)
feats["bilirrubina_alta"] = (f["Bilirubin_total"] > 1.2).astype(np.int8)
feats["BUN_creatinina"] = (f["BUN"] / f["Creatinine"]).astype(np.float32)

# Datos del paciente
feats["Age"] = df["Age"]
feats["Gender"] = df["Gender"]
feats["Unit1"] = df["Unit1"].fillna(-1)
feats["Unit2"] = df["Unit2"].fillna(-1)
feats["HospAdmTime"] = df["HospAdmTime"]
feats["ICULOS"] = df["ICULOS"]
del filled, f

## 6. Armar la matriz final y dividir en entrenamiento / validación / prueba

- **Hospital A → 80 % entrenamiento, 20 % validación.** La división es **por paciente** (todas las horas de un paciente van al mismo grupo). Si se dividiera por fila, el modelo "conocería" al paciente y parecería mejor de lo que es (*fuga de datos*).
- La división respeta la proporción de pacientes con sepsis (estratificada).
- **Hospital B → solo prueba final.** El modelo nunca lo ve al entrenar: así demostramos que **funciona en otro hospital**.

In [6]:
F = pd.DataFrame(feats)
F = F.astype({c: np.float32 for c in F.columns if F[c].dtype == np.float64})
FEATURES = list(F.columns)
X = pd.concat([df[["patient_id", "hospital", "hour"]], F, df[["SepsisLabel"]]], axis=1)
del feats, F

pat = X.groupby("patient_id", observed=True).agg(hospital=("hospital", "first"), sepsis=("SepsisLabel", "max"))
rng = np.random.default_rng(42)
split = pd.Series("test", index=pat.index)
for s in [0, 1]:
    ids = pat[(pat["hospital"] == "A") & (pat["sepsis"] == s)].index.to_numpy()
    rng.shuffle(ids)
    n_val = int(round(0.2 * len(ids)))
    split[ids[:n_val]] = "val"
    split[ids[n_val:]] = "train"
X["split"] = X["patient_id"].map(split).astype("category")

resumen = X.groupby("split", observed=True).agg(
    pacientes=("patient_id", "nunique"), filas=("hour", "size"), pct_horas_sepsis=("SepsisLabel", "mean"))
resumen["pct_horas_sepsis"] = (100 * resumen["pct_horas_sepsis"]).round(2)
resumen["pacientes_con_sepsis"] = X[X["SepsisLabel"] == 1].groupby("split", observed=True)["patient_id"].nunique()
print(f"Features: {len(FEATURES)} | Memoria: {X.memory_usage(deep=True).sum() / 1024**2:,.0f} MB")
resumen

Features: 140 | Memoria: 668 MB


/tmp/ipykernel_1622/2175076495.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X["split"] = X["patient_id"].map(split).astype("category")


,pacientes,filas,pct_horas_sepsis,pacientes_con_sepsis
split,,,,
test,20000,761995,1.41,1142
train,16269,631994,2.17,1432
val,4067,158221,2.16,358


## 7. Control de calidad

1. Ningún paciente aparece en dos grupos (no hay fuga entre entrenamiento, validación y prueba).
2. Ninguna feature usa el futuro: la primera hora de cada paciente no puede tener cambio de 6 h.
3. ¿Hay señal? Comparamos algunas features entre horas sin sepsis (0) y con sepsis (1).

In [7]:
assert X.groupby("patient_id", observed=True)["split"].nunique().max() == 1, "Paciente en dos grupos"
assert X.loc[X["hour"] == 0, "HR_6h_cambio"].notna().sum() == 0, "Hora 0 con cambio de 6h"
print("OK: sin fuga entre grupos y sin datos del futuro en la hora 0")

check = ["HR", "Resp", "Temp", "MAP", "indice_shock", "SIRS", "qSOFA", "HR_6h_cambio", "Lactate", "ICULOS"]
X[X["split"] == "train"].groupby("SepsisLabel")[check].mean().T.round(2)

OK: sin fuga entre grupos y sin datos del futuro en la hora 0


SepsisLabel,0,1
HR,84.82,90.57
Resp,18.76,20.80
Temp,36.94,37.18
MAP,78.91,78.44
indice_shock,0.73,0.78
SIRS,1.16,1.63
qSOFA,0.42,0.59
HR_6h_cambio,-0.13,1.13
Lactate,1.93,1.76
ICULOS,26.77,56.08


## 8. Guardar para los modelos

- `features.pkl`: la matriz completa (`float32`) con `patient_id`, `hospital`, `hour`, `split` y `SepsisLabel`.
- `feature_list.json`: el orden exacto de las columnas que entran al modelo (el `scorer` y la red de Jhojan deben usar el mismo orden).

**Nota para la red neuronal (Jhojan):** las features tienen valores vacíos (`NaN`) cuando algo nunca se midió.
XGBoost los maneja solo; para la red hay que rellenarlos (por ejemplo con la mediana de *train*) y estandarizar, usando solo estadísticas de *train*.

In [8]:
X.to_pickle(f"{BASE}/features.pkl")
with open(f"{BASE}/feature_list.json", "w") as fh:
    json.dump(FEATURES, fh, indent=1)
print(f"Guardado: {BASE}/features.pkl ({os.path.getsize(f'{BASE}/features.pkl') / 1024**2:,.0f} MB)")
print(f"Guardado: {BASE}/feature_list.json ({len(FEATURES)} features)")
print(f"Tiempo total del notebook: {time.time() - t_start:.1f} s en {platform.machine()}")

Guardado: /home/ibm-user/shared/sepsis/data/processed/features.pkl (665 MB)
Guardado: /home/ibm-user/shared/sepsis/data/processed/feature_list.json (140 features)
Tiempo total del notebook: 17.7 s en s390x
